# 03 — Quantile Learn-Then-Test (QLTT)

This notebook illustrates a simple implementation of **Quantile Learn-Then-Test (QLTT)**.

Rather than controlling the **average** risk, QLTT aims to control a chosen
**quantile** of the risk distribution (e.g., a 0.1-quantile, interpreted as
an "outage" risk for the worst 10% of episodes).

We will:

1. Generate a synthetic CSV `../data/sample_QLTT_losses.csv` if missing.
2. Explain its format (multiple loss values per hyperparameter).
3. Implement a basic QLTT pipeline for a user-specified quantile `q`:
   - For each hyperparameter λ, we have a set of losses (one per episode).
   - We use a quantile confidence bound (Howard–Ramdas style) to construct a
     one-sided confidence interval for the q-quantile of the risk.
   - We convert this to a p-value for the null `H_λ: R_q(λ) > α`.
   - We apply the Bonferroni correction to ensure FWER control at level `δ`.
4. Inspect which hyperparameters are declared reliable at level `α, q, δ`.


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path("..").resolve()
DATA_DIR = BASE_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

QLTT_CSV = DATA_DIR / "sample_QLTT_losses.csv"

print("Base directory:", BASE_DIR)
print("Data directory:", DATA_DIR)
print("QLTT CSV path:", QLTT_CSV)


In [ ]:
def generate_sample_qltt_csv(path: Path, num_lambdas: int = 20, num_episodes: int = 120, seed: int = 2):
    """Generate a synthetic CSV for QLTT.

    Format:
    - One row per hyperparameter configuration λ.
    - Columns:
        - 'lambda_id' (int)
        - 'loss_1..loss_M'  (per-episode loss, e.g., average delay for that episode)
    """
    rng = np.random.default_rng(seed)
    lambda_ids = np.arange(num_lambdas)

    # Construct configs with different tails: some have light tails (more stable),
    # others have heavier tails (occasional large losses).
    base_means = np.linspace(5.0, 15.0, num_lambdas)  # e.g., mean delay in ms
    base_scales = np.linspace(1.0, 4.0, num_lambdas)

    losses = []
    for mu, scale in zip(base_means, base_scales):
        # Truncated normal-like via clipping
        row = np.clip(rng.normal(loc=mu, scale=scale, size=num_episodes), 0.0, 30.0)
        losses.append(row)

    loss_arr = np.stack(losses, axis=0)

    cols = ["lambda_id"] + [f"loss_{i+1}" for i in range(num_episodes)]
    data = np.column_stack([lambda_ids, loss_arr])
    df = pd.DataFrame(data, columns=cols)
    df["lambda_id"] = df["lambda_id"].astype(int)

    df.to_csv(path, index=False)
    return df


if not QLTT_CSV.exists():
    print("CSV does not exist — generating synthetic sample_QLTT_losses.csv ...")
    df_qltt = generate_sample_qltt_csv(QLTT_CSV)
else:
    print("CSV already exists — loading:", QLTT_CSV)
    df_qltt = pd.read_csv(QLTT_CSV)

print("\nFirst few rows of the QLTT CSV:")
display(df_qltt.head())

print("\nColumns:")
print(df_qltt.columns.tolist())


In [ ]:
lambda_ids = df_qltt["lambda_id"].values
loss_cols = [c for c in df_qltt.columns if c.startswith("loss_")]
loss_mat = df_qltt[loss_cols].values  # (L, M)

num_lambdas, num_episodes = loss_mat.shape
print(f"Number of hyperparameters: {num_lambdas}")
print(f"Number of episodes per hyperparameter: {num_episodes}")

# Empirical mean and quantile just for visualization
mean_loss = loss_mat.mean(axis=1)
q = 0.1  # outage rate
empirical_q = np.quantile(loss_mat, 1 - q, axis=1)

plt.figure()
plt.plot(lambda_ids, mean_loss, "o-", label="mean loss")
plt.plot(lambda_ids, empirical_q, "s-", label=f"{1-q:.1f}-quantile (empirical)")
plt.xlabel("lambda_id")
plt.ylabel("Loss")
plt.title("Mean and high quantile per hyperparameter")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
def R_q_star(losses, q, eps):
    """Compute the upper confidence bound R_q*(λ, eps) for the q-quantile.
    
    Based on Howard & Ramdas (2022)-style bound as used in the QLTT paper.
    losses: 1D array of losses for a fixed λ.
    q: outage rate in [0,1]. We control the (1-q)-quantile.
    eps: confidence level for this λ (between 0 and 1).
    """
    losses = np.asarray(losses, float)
    n = len(losses)
    if n < 5:
        # Small-sample fallback: just use max as conservative bound
        return float(np.max(losses))

    # r_n term (Theorem 1 of Howard & Ramdas type)
    rn = 1.4 * np.log(np.log(2.1 * n)) + np.log(10.0 / eps)
    # Adjusted quantile level q*
    q_star = q - 1.5 * np.sqrt(q * (1 - q) * rn / n) - 0.8 * rn / n
    # Clip q_star to reasonable range
    q_star = float(np.clip(q_star, 0.0, 1.0))

    # We care about the (1 - q_star)-quantile of losses
    sorted_losses = np.sort(losses)
    idx = int(np.floor(n * (1.0 - q_star))) - 1
    idx = max(0, min(idx, n - 1))
    return float(sorted_losses[idx])


def qltt_p_value(losses, alpha, q, grid_size: int = 200):
    """Approximate p-value for H_λ: R_q(λ) > alpha.

    We search for the smallest eps in a grid such that R_q_star(losses, q, eps) >= alpha.
    The p-value is then that smallest eps (or 1.0 if none found).
    """
    # If empirical quantile is already well above alpha, p-value will be large anyway,
    # but we still run the grid for simplicity.
    eps_grid = np.logspace(-4, 0, grid_size)  # from 1e-4 to 1
    candidate_eps = []
    for eps in eps_grid:
        ub = R_q_star(losses, q, eps)
        if ub >= alpha:
            candidate_eps.append(eps)
            break  # first such eps is the inf over this grid
    if not candidate_eps:
        return 1.0
    return float(candidate_eps[0])


In [ ]:
alpha = 10.0  # e.g., max acceptable 0.9-quantile of delay in ms
q = 0.1       # outage rate, we control (1-q)-quantile
delta = 0.1   # FWER target

print(f"Target alpha (quantile level): {alpha}\nOutage rate q: {q}\nDelta (FWER): {delta}\n")

p_values = []
for i in range(num_lambdas):
    losses_i = loss_mat[i, :]
    p_i = qltt_p_value(losses_i, alpha=alpha, q=q)
    p_values.append(p_i)

p_values = np.array(p_values)
bonf_threshold = delta / num_lambdas

selected = p_values < bonf_threshold
selected_lambdas = lambda_ids[selected]

results_df = pd.DataFrame({
    "lambda_id": lambda_ids,
    "mean_loss": mean_loss,
    f"emp_{1-q:.1f}_quantile": empirical_q,
    "p_value": p_values,
    "selected_by_QLTT_Bonferroni": selected,
}).sort_values(f"emp_{1-q:.1f}_quantile")

print(f"Bonferroni threshold: {bonf_threshold:.4g}\n")
print("Hyperparameters sorted by empirical high quantile:")
display(results_df.head(10))

print("\nHyperparameters selected by QLTT + Bonferroni:", selected_lambdas.tolist())
